# Open a CZI microscopy file with napari

This notebook installs/checks the required packages, opens a CZI file with `aicsimageio` using the `aicspylibczi` backend, prints useful metadata, and displays each channel as a separate napari layer. If napari cannot open in the current environment, each channel is saved as a TIFF fallback.

## 1. Install or verify dependencies

The project already lists these packages in `pyproject.toml`, but this cell makes the notebook portable by installing anything missing in the active Jupyter kernel.

In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys

required_packages = [
    ("aicsimageio", "aicsimageio>=4.14.0"),
    ("aicspylibczi", "aicspylibczi"),
    ("napari", "napari[all]"),
    ("jupyter_rfb", "jupyter_rfb"),
    ("tifffile", "tifffile"),
    ("numpy", "numpy"),
]

packages_to_install = []
for import_name, pip_spec in required_packages:
    if importlib.util.find_spec(import_name) is None:
        packages_to_install.append(pip_spec)

try:
    from packaging.version import Version
    if importlib.util.find_spec("aicsimageio") is not None and Version(metadata.version("aicsimageio")) < Version("4.14.0"):
        packages_to_install.append("aicsimageio>=4.14.0")
except Exception:
    pass

if packages_to_install:
    print("Installing/upgrading packages:", ", ".join(packages_to_install))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", *packages_to_install])
    print("If a package was upgraded, restart the kernel before running the remaining cells.")
else:
    print("All required packages are already available in this kernel.")

## 2. Configure Qt support for napari in Jupyter

napari is a Qt application. In notebooks, `%gui qt` lets the Qt event loop run without blocking the kernel. This is safe to run in Jupyter and is skipped automatically outside IPython.

In [ ]:
try:
    ip = get_ipython()
    ip.run_line_magic("gui", "qt")
    print("Enabled Qt event loop with %gui qt.")
except NameError:
    print("Not running inside IPython/Jupyter; Qt setup will happen when napari opens.")

## 3. Load the CZI file with AICSImage

This uses `aicspylibczi` as the CZI backend. A small compatibility shim is included for environments where old `aicsimageio` expects `CziFile.dims_shape()` but newer `aicspylibczi` exposes the same functionality as `CziFile.get_dims_shape()`.

In [ ]:
from pathlib import Path

import aicspylibczi  # Confirms the intended CZI backend is installed; do not use czifile.
from aicspylibczi import CziFile
from aicsimageio import AICSImage

# Compatibility for aicsimageio 3.x with newer aicspylibczi versions.
# Old AICSImage/CziReader calls czi.dims_shape(), while current aicspylibczi uses get_dims_shape().
if not hasattr(CziFile, "dims_shape") and hasattr(CziFile, "get_dims_shape"):
    CziFile.dims_shape = CziFile.get_dims_shape
    print("Patched aicspylibczi.CziFile.dims_shape for this notebook kernel.")

CZI_PATH = Path("/Users/serenasritharan/Projects/biochemical-stain-assessment/data_1_5/JH-311/AbPAS/ITG_Rusha_JH-311_Organoid_AbPAS_ACAN4_F1.czi")

if not CZI_PATH.exists():
    raise FileNotFoundError(f"CZI file not found: {CZI_PATH}")

try:
    from aicsimageio.readers import CziReader
    image = AICSImage(str(CZI_PATH), reader=CziReader)
    print("Loaded CZI with AICSImage and explicit CziReader/aicspylibczi backend.")
except (ImportError, TypeError):
    image = AICSImage(str(CZI_PATH))
    print("Loaded CZI with AICSImage; aicspylibczi is installed and available as the CZI backend.")

## 4. Print image metadata

The metadata below gives the raw array shape, dimension labels, available channel names, and physical pixel sizes when present in the CZI metadata.

In [ ]:
def dims_order(aics_image):
    dims = aics_image.dims
    if hasattr(dims, "order"):
        return dims.order
    if hasattr(dims, "dimensions"):
        return "".join(dims.dimensions)
    return str(dims)


def dims_sizes(aics_image):
    dims = aics_image.dims
    if hasattr(dims, "shape") and hasattr(dims, "order"):
        return dict(zip(dims.order, dims.shape))
    if hasattr(dims, "dims") and hasattr(dims, "shape"):
        return dict(zip(dims.dims, dims.shape))
    if isinstance(dims, str) and hasattr(aics_image, "shape"):
        return dict(zip(dims, aics_image.shape))
    return {}


def channel_names(aics_image):
    names = getattr(aics_image, "channel_names", None)
    if names:
        return list(names)
    if hasattr(aics_image, "get_channel_names"):
        names = aics_image.get_channel_names()
        if names:
            return list(names)
    sizes = dims_sizes(aics_image)
    channel_count = sizes.get("C", 1)
    return [f"Channel {idx}" for idx in range(channel_count)]


def physical_pixel_sizes(aics_image):
    sizes = getattr(aics_image, "physical_pixel_sizes", None)
    if sizes is not None:
        return sizes
    if hasattr(aics_image, "get_physical_pixel_size"):
        return aics_image.get_physical_pixel_size()
    return None


print("Shape:", image.shape)
print("Dims:", dims_order(image))
print("Dim sizes:", dims_sizes(image))
print("Channel names:", channel_names(image))
print("Physical pixel sizes:", physical_pixel_sizes(image))

## 5. Prepare one array per channel

For viewing, this selects the first index of any non-spatial, non-channel dimensions such as time, scene, or mosaic dimensions, while preserving `Z`, `Y`, and `X` when present.

In [ ]:
import numpy as np

order = dims_order(image)
sizes = dims_sizes(image)
names = channel_names(image)
channel_count = sizes.get("C", len(names) if names else 1)

target_order = "ZYX" if "Z" in order else "YX"
fixed_indices = {dim: 0 for dim in order if dim not in set(target_order + "C")}

channel_arrays = []
for channel_index in range(channel_count):
    if "C" in order:
        data = image.get_image_data(target_order, C=channel_index, **fixed_indices)
        layer_name = names[channel_index] if channel_index < len(names) else f"Channel {channel_index}"
    else:
        data = image.get_image_data(target_order, **fixed_indices)
        layer_name = names[0] if names else "Image"

    data = np.squeeze(data)
    channel_arrays.append((layer_name, data))

print(f"Prepared {len(channel_arrays)} napari layer(s):")
for layer_name, data in channel_arrays:
    print(f"- {layer_name}: shape={data.shape}, dtype={data.dtype}")

## 6. Open napari, or save TIFF fallback files

This cell creates a napari viewer and adds each prepared channel as a separate image layer. If Qt/napari cannot start in the current notebook environment, the exception is caught and each channel is written to `napari_tiff_fallback/` as a TIFF file.

In [ ]:
import re

import tifffile


def safe_filename(name):
    cleaned = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name)).strip("_")
    return cleaned or "channel"


try:
    import napari

    viewer = napari.Viewer(title=CZI_PATH.name)
    for layer_name, data in channel_arrays:
        is_rgb = data.ndim >= 3 and data.shape[-1] in (3, 4)
        viewer.add_image(data, name=layer_name, rgb=is_rgb)

    print("Opened napari viewer and added all channels as separate layers.")
    print("If the window does not appear, check that the notebook kernel has a Qt-capable GUI environment.")

except Exception as exc:
    fallback_dir = CZI_PATH.with_name("napari_tiff_fallback")
    fallback_dir.mkdir(parents=True, exist_ok=True)
    print(f"napari failed to open: {type(exc).__name__}: {exc}")
    print(f"Saving TIFF fallback files to: {fallback_dir}")

    for layer_name, data in channel_arrays:
        out_path = fallback_dir / f"{safe_filename(layer_name)}.tif"
        tifffile.imwrite(out_path, data)
        print(f"Saved {out_path}")